# GRACE H1 — Bắt đầu full pipeline tại đây

Đây là notebook điều phối cho **local → Kaggle → Colab CPU → local**. Mở notebook này trước và chạy từ trên xuống. Khi cần chuyển môi trường, cell dừng bằng RuntimeError có hướng dẫn; tải output về bundles/ rồi Run All lại. Không cần đoán cell nào được chạy tiếp.

| Bước | Chạy ở đâu | File cần mang đi / mang về |
|---|---|---|
| 1. Train projection CE + SupCon/hard | Kaggle GPU | training-inputs.zip → trained-heads.zip |
| 2. Whitening, retrieval, Joern, rerank, frozen prompts | Colab CPU, accelerator None | cpu-inputs.zip + joern-runtime.zip + trained-heads.zip → cpu-prepared.zip |
| 3. Preflight, Vertex LLM, metrics/paired CI | Local | Import heads + prepared; dùng response cache/ADC đã có |

Mặc định full test Devign, hai arm baseline/H1. Thêm ReVeal bằng configs/experiment.json trước khi xuất bundle và giữ cùng config ở mọi môi trường. Không cần copy lại embedding/CodeT5 weights thủ công.

- Kaggle: [kaggle_train_projection.ipynb](./kaggle_train_projection.ipynb).
- Colab: [colab_cpu_prepare.ipynb](./colab_cpu_prepare.ipynb).
- Hướng dẫn chuyển file: [CLOUD_HANDOFF.md](../docs/CLOUD_HANDOFF.md).

In [8]:
from pathlib import Path
import os,sys,json,subprocess,importlib.util
candidates=([Path(os.environ['GRACE_H1_ROOT'])] if os.environ.get('GRACE_H1_ROOT') else [])
candidates += [Path.cwd(),*Path.cwd().parents,Path('D:/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection'),Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection')]
ROOT=next((p.resolve() for p in candidates if (p/'grace_h1').is_dir()),None)
if ROOT is None: raise RuntimeError('Set GRACE_H1_ROOT to the local project folder')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
INSTALL_DEPENDENCIES=True
TRAIN_ON_KAGGLE=True
PREPARE_ON_COLAB=True
RUN_INFERENCE=True  # Calls Vertex for prompts missing from cache, only after all handoffs pass.
HEADS_BUNDLE=ROOT/'bundles'/'grace-h1-trained-heads.zip'
PREPARED_BUNDLE=ROOT/'bundles'/'grace-h1-cpu-prepared.zip'
config=json.loads((ROOT/'configs'/'experiment.json').read_text(encoding='utf-8'))
print('Project:',ROOT)
print(json.dumps(config,indent=2))

Project: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection
{
  "datasets": [
    "devign"
  ],
  "splits": [
    "test"
  ],
  "objectives": [
    "ce",
    "supcon_hard"
  ],
  "llm_arms": [
    "baseline_whitening",
    "h1_supcon_hard"
  ],
  "training": {
    "dimension": 128,
    "epochs": 30,
    "batch_size": 256,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "supcon_weight": 0.1,
    "hard_negative_weight": 0.1,
    "temperature": 0.1,
    "margin": 0.2,
    "class_weighted_ce": true,
    "patience": 6,
    "seed": 42,
    "device": "auto"
  },
  "pipeline": {
    "semantic_top_k": 5,
    "lexical_weight": 0.7,
    "whitening_dimension": 256,
    "graph_workers": 2,
    "joern_home": null
  },
  "inference_workers": 4,
  "bootstrap": 1000
}


In [9]:
if INSTALL_DEPENDENCIES:
    if TRAIN_ON_KAGGLE:
        # Local handoff/inference does not need PyTorch or CUDA.
        subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-cpu.txt'),'google-genai>=1.0,<3'])
    else:
        if importlib.util.find_spec('torch') is None:
            subprocess.check_call([sys.executable,'-m','pip','install','torch>=2.6,<3','--index-url','https://download.pytorch.org/whl/cpu'])
        subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements.txt')])
from grace_h1.io import read_json,load_local_environment
from grace_h1.portable import create_bundle,import_bundle
from grace_h1.offload import require_trained_heads,require_prepared_runs
from grace_h1.experiment import train_all,prepare_all,preflight_all,infer_all,evaluate_all
import pandas as pd
load_local_environment(ROOT)

## Bước 1 — Head trên Kaggle

**Lần đầu:** cell dưới tạo training-inputs bundle rồi dừng. Đính kèm bundle vào Kaggle Dataset, upload kaggle_train_projection.ipynb, bật GPU/Internet, Run All. Tải grace-h1-trained-heads.zip về bundles/ local rồi Run All notebook này lại.

**Nếu đã có head khớp:** import/reuse, không train trên local. Head/config/source/data sai fingerprint sẽ dừng để xuất lại đúng bộ. TRAIN_ON_KAGGLE=False chỉ dùng khi chủ động muốn train local.

In [10]:
if HEADS_BUNDLE.exists(): print(import_bundle(ROOT,HEADS_BUNDLE,'trained_heads'))
if TRAIN_ON_KAGGLE:
    try:
        selection=require_trained_heads(ROOT,config)
    except (FileNotFoundError,KeyError,ValueError) as exc:
        info=create_bundle(ROOT,ROOT/'bundles'/'grace-h1-training-inputs.zip','training_inputs')
        print(info)
        raise RuntimeError('DỪNG BƯỚC 1: mở notebooks/kaggle_train_projection.ipynb trên Kaggle; đính kèm training-inputs.zip; tải trained-heads.zip về bundles/ rồi Run All lại. Chi tiết head: '+str(exc)) from exc
else:
    selection=train_all(ROOT,config)
    print(create_bundle(ROOT,HEADS_BUNDLE,'trained_heads'))
display(pd.DataFrame([{'dataset':d,'objective':o,'parameters':h['parameters'],'best_epoch':h['best']['epoch'],
                      'validation_mcc':h['best']['metrics']['mcc'],'trained_device':h['device']}
                     for d,heads in selection['heads'].items() for o,h in heads.items()]))

{'kind': 'trained_heads', 'copied': 0, 'reused_identical': 11, 'archive_sha256': 'c10fe71837d59a331105a9329c450705e210763d51fa2c9cffe3e2f23cba056f'}


,dataset,objective,parameters,best_epoch,validation_mcc,trained_device
0,devign,ce,98690,16,0.236823,cuda
1,devign,supcon_hard,98690,13,0.222189,cuda


## Bước 2 — CPU/RAM trên Colab

**Lần đầu:** cell dưới tạo CPU-inputs bundle rồi dừng. Upload ba zip vào MyDrive/GRACE-H1-transfer: cpu-inputs, joern-runtime và trained-heads. Upload colab_cpu_prepare.ipynb, chọn accelerator **None**, Run All.

Colab fit whitening, transform vector, common top50 → top5, tái dùng graph/sinh graph thiếu, lexical/AST rerank và đóng băng prompt. Colab không có credential và không gọi LLM. Checkpoint cache được lưu về Drive mỗi 500 graph và cuối arm.

**Quay về:** tải grace-h1-cpu-prepared.zip vào bundles/ local, giữ trained-heads.zip ở đó, Run All lại notebook này. Cell kiểm tra fingerprint/freeze rồi tiếp tục, không chạy preparation local. PREPARE_ON_COLAB=False chỉ dùng khi chủ động muốn prepare local với Joern.

In [11]:
if PREPARE_ON_COLAB:
    if PREPARED_BUNDLE.exists(): print(import_bundle(ROOT,PREPARED_BUNDLE,'cpu_prepared'))
    try:
        run_rows=require_prepared_runs(ROOT,config)
    except (FileNotFoundError,KeyError,ValueError) as exc:
        info=create_bundle(ROOT,ROOT/'bundles'/'grace-h1-cpu-inputs.zip','cpu_inputs')
        print(info)
        runtime=ROOT/'bundles'/'grace-h1-joern-runtime.zip'
        print('Joern runtime:',runtime,'exists:',runtime.exists())
        raise RuntimeError('DỪNG BƯỚC 2: mở notebooks/colab_cpu_prepare.ipynb trên Colab CPU; upload cpu-inputs.zip, joern-runtime.zip, trained-heads.zip; tải cpu-prepared.zip về bundles/ rồi Run All lại. Chi tiết prepare: '+str(exc)) from exc
else:
    run_rows=prepare_all(ROOT,config)
display(pd.DataFrame(run_rows))

{'kind': 'cpu_prepared', 'copied': 415, 'reused_identical': 0, 'archive_sha256': '4fb2ced52087a556c3bcea231cc9b86667f08d58edb6f516288cf2cb7e5dd96e'}


,arm,dataset,path,split
0,baseline_whitening,devign,runs/devign/test/baseline_whitening/9afe326872...,test
1,h1_supcon_hard,devign,runs/devign/test/h1_supcon_hard/7c4a52705268af6f,test


## Bước 3 — Local: preflight trước khi gọi Vertex

Đã có head và frozen prompts từ cloud. ADC/Vertex chỉ dùng tại local. Exact prompt/model/decoding khớp sẽ reuse response cũ; tổng new requests từng arm là cận trên vì prompt trùng giữa các arm tiếp tục được reuse khi chạy tuần tự. RUN_INFERENCE=False chỉ đọc cache.

RUN_INFERENCE=True ở cell cấu hình: khi chạy tiếp cell inference bên dưới sẽ gọi API cho phần thiếu.

In [12]:
plans=preflight_all(ROOT,config)
display(pd.DataFrame([{k:p[k] for k in ['dataset','split','arm','queries','cached_queries','unique_prompts','new_unique_requests','worst_case_attempts']} for p in plans]))

,dataset,split,arm,queries,cached_queries,unique_prompts,new_unique_requests,worst_case_attempts
0,devign,test,baseline_whitening,2732,2162,2731,569,2845
1,devign,test,h1_supcon_hard,2732,2008,2730,724,3620


In [13]:
results=infer_all(ROOT,config,allow_api=RUN_INFERENCE)
display(pd.DataFrame([{k:r.get(k) for k in ['dataset','split','arm','status','cached_queries','new_unique_requests','api_attempts_this_run','request_error_queries','wall_seconds']} for r in results]))

LLM unique requests 25/569
LLM unique requests 50/569
LLM unique requests 75/569
LLM unique requests 100/569
LLM unique requests 125/569
LLM unique requests 150/569
LLM unique requests 175/569
LLM unique requests 200/569
LLM unique requests 225/569
LLM unique requests 250/569
LLM unique requests 275/569
LLM unique requests 300/569
LLM unique requests 325/569
LLM unique requests 350/569
LLM unique requests 375/569
LLM unique requests 400/569
LLM unique requests 425/569
LLM unique requests 450/569
LLM unique requests 475/569
LLM unique requests 500/569
LLM unique requests 525/569
LLM unique requests 550/569
LLM unique requests 569/569
LLM unique requests 25/463
LLM unique requests 50/463
LLM unique requests 75/463
LLM unique requests 100/463
LLM unique requests 125/463
LLM unique requests 150/463
LLM unique requests 175/463
LLM unique requests 200/463
LLM unique requests 225/463
LLM unique requests 250/463
LLM unique requests 275/463
LLM unique requests 300/463
LLM unique requests 325/46

,dataset,split,arm,status,cached_queries,new_unique_requests,api_attempts_this_run,request_error_queries,wall_seconds
0,devign,test,baseline_whitening,complete,2162,569,569,0,269.064223
1,devign,test,h1_supcon_hard,complete,2269,463,463,0,205.837933


## Bước 4 — Metrics và paired comparison

Phần tổng hợp nhỏ chạy local sau LLM. F1/MCC trên valid predictions, luôn báo invalid rate và recall trên mọi positive. Paired bootstrap dùng cùng query valid ở hai arm. Chưa có kết quả H1 mới trong notebook bàn giao.

In [14]:
report=evaluate_all(ROOT,config)
rows=[]
for r in report['scores']:
    m=r['metrics'];i=r['inference']
    rows.append({'dataset':r['dataset'],'split':r['split'],'arm':r['arm'],'n':m['n'],'f1_valid':m['f1'],
                 'mcc_valid':m['mcc'],'recall_all_positives':m['recall_all_positives'],'invalid_rate':m['invalid_rate'],
                 'new_requests':i['new_unique_requests'],'api_attempts':i['api_attempts_this_run'],
                 'new_usage':i['provider_usage_new_requests']})
display(pd.DataFrame(rows))
print('Status:',report['status'])
for comparison in report['comparisons']: print(json.dumps(comparison,indent=2))
print('Saved:',ROOT/'runs'/'experiment_summary.json')

,dataset,split,arm,n,f1_valid,mcc_valid,recall_all_positives,invalid_rate,new_requests,api_attempts,new_usage
0,devign,test,baseline_whitening,2732,0.357328,0.058664,0.274900,0.0,569,569,"{'candidates_token_count': 1138, 'prompt_token..."
1,devign,test,h1_supcon_hard,2732,0.343268,0.028007,0.266135,0.0,463,463,"{'candidates_token_count': 926, 'prompt_token_..."


Status: complete
{
  "common_valid_n": 2732,
  "excluded_invalid_pair_n": 0,
  "delta_right_minus_left": {
    "f1": -0.014059566876350238,
    "mcc": -0.030656958313920422,
    "recall": -0.008764940239043784,
    "precision": -0.02699757661908392
  },
  "ci95_query_bootstrap": {
    "f1": [
      -0.029991025376855506,
      0.0012248143540481051
    ],
    "mcc": [
      -0.051260810709317844,
      -0.009035736117492449
    ],
    "recall": [
      -0.02301873370782389,
      0.0054224691004614075
    ],
    "precision": [
      -0.04536714885779902,
      -0.008761080975809982
    ]
  },
  "bootstrap_repetitions": 1000,
  "note": "query bootstrap; repeated code/project dependence is not cluster-adjusted",
  "left": "runs/devign/test/baseline_whitening/9afe3268729648c6",
  "right": "runs/devign/test/h1_supcon_hard/7c4a52705268af6f"
}
Saved: /mnt/d/JUNIOR/LabThayTho/GRACE-H1-SecurityProjection/runs/experiment_summary.json
